# Lab 4 - Grounded Answers with Citations (Track N: notebook)

One file, top to bottom. Work the cells in order, run each one, read its
output.

This repo has two tracks. **This notebook is Track N.** The alternative is
Track S - the scripts `labs/lab4/rag.py` + `labs/lab4/evaluate.py`. Pick one
and ignore the other for the whole lab. (You still submit those two files +
`report.md` either way.)

| Part | What you do | ~time |
|---|---|---|
| 0 | Environment + Lab 3 retriever check | 10 min |
| A | Write `ANSWER_SYSTEM` yourself | 20 min |
| B | Citation validation + the repair strategy | 30 min |
| C | Refusal, both directions | 30 min |
| D | The judge: rubrics + calibration (κ) | 45 min |
| E | Full eval + gold-context decomposition | 35 min |
| Submit | Copy into `rag.py`/`evaluate.py` + fill `report.md` | 10 min |

**Prerequisite:** Lab 3 finished; you know your winning retrieval config.

**Keep open:** `labs/lab4/CONCEPTS.md`.

**The one rule:** run cells in order, and when a cell says *your turn*, edit it
there. Numbered TODOs (`A1`, `B2.1`, ...) come with scaffold lines - fill them
in; the `assert` tells you what is still missing. When you are done, copy your
work into `labs/lab4/rag.py` and `labs/lab4/evaluate.py` - those are the graded
files - and fill `report.md` by hand. Never regenerate the report.

**Targets (45 questions):** citation validity **1.00** (code guarantee) ·
faithfulness ≥ 0.90 · correctness ≥ 0.75 · refusal recall ≥ 4/5 · refusal
precision ≥ 0.70 · κ ≥ 0.4 for both rubrics.

**Both refusal numbers are noisy** (n = 5 unanswerable). Report raw counts
beside the ratios. **Never report only one direction** - refuse everything and
recall is 1.00.

**The war story:** a truncated judge verdict scored as 0 once turned a true
faithfulness of 0.933 into a reading of 0.667. **A parse failure is missing
data, not a failing answer.**


## 0 - Setup


In [ ]:
# Make the shared library importable and load settings. Safe to re-run.
import os, sys, json, re, statistics
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "aip").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "aip").is_dir():
    ROOT = Path.cwd().parents[1]          # notebook opened from labs/lab4
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
os.chdir(ROOT)
print("repo root:", ROOT)
print("python:", sys.version.split()[0])


In [ ]:
# Dependencies + API key.
ok = True
for mod in ("pydantic", "httpx", "dotenv", "numpy"):
    try:
        __import__(mod)
        print(f"PASS  import {mod}")
    except ImportError:
        ok = False
        print(f"FAIL  import {mod}  ->  pip install -r requirements.txt")

key = os.getenv("GEMINI_API_KEY", "")
print(("PASS" if key.strip() else "FAIL"), "GEMINI_API_KEY set")
assert ok and key.strip(), "fix the FAIL lines above before continuing"


In [ ]:
# Lab 3 must be finished: build_retriever() is YOUR winning chunking config.
# Also load the golden set: 45 questions, 5 unanswerable (Q36-Q40).
from labs.lab3.search import build_chunks, load_corpus, load_questions
from labs.lab4.rag import Answer, REFUSAL, is_refusal, _cited_indices

corpus = load_corpus()
questions_all = load_questions(include_unanswerable=True)   # 45
questions_answerable = load_questions()                     # 40
unanswerable = [q for q in questions_all if q["kind"] == "unanswerable" or not q["relevant_docs"]]
print(f"corpus: {len(corpus)} docs")
print(f"golden: {len(questions_all)} rows -> {len(questions_answerable)} answerable, {len(unanswerable)} unanswerable")
print("unanswerable ids:", [q["id"] for q in unanswerable])
assert len(questions_all) == 45 and len(questions_answerable) == 40


### Read `labs/lab4/rag.py` structure (not the answers)

Open the file and note the **provided** pieces before you write anything:

- `REFUSAL` - the exact string downstream code matches. Do not paraphrase it.
- `Answer` - the dataclass your pipeline returns. Its fields are the contract.
- `is_refusal`, `_cited_indices`, `_generate`, `_repair_note` - helpers you call.

Your TODOs: **A** (`ANSWER_SYSTEM`), **B2** (`validate_answer`),
**B3** (repair strategy inside `answer_question`), **E2**
(`answer_with_gold_context`).

In `evaluate.py`: **D1** (two judges) and the **build_retriever** config.


## Part A - The generation prompt (20 min)

**Write `ANSWER_SYSTEM` yourself, before opening `aip/rag.py`.** Six required
elements (T4 §6.1). Then diff yours against the reference.

> Rule 6 (length) is not cosmetic. Output tokens dominate latency and cost.


In [ ]:
# YOUR TURN - A1: write ANSWER_SYSTEM. Six required elements (T4 §6.1):
#
#   A1.1  Answer ONLY from numbered sources; forbid general knowledge
#   A1.2  Cite by index: [1], [2][5]
#   A1.3  Never cite a number that was not supplied
#   A1.4  Exact refusal string, given VERBATIM (use REFUSAL from rag.py)
#   A1.5  Sources disagree -> surface it, do not pick silently
#   A1.6  Length discipline: two or three sentences unless more is needed
#
# Write it as the system prompt that will be sent with every generation.
# Checkpoint: all six elements present; refusal string matches REFUSAL exactly.

ANSWER_SYSTEM = f"""\
# TODO A1.1-A1.6: write the system prompt here.
# Must include the exact REFUSAL string:
#   {REFUSAL}
"""

# Auto-check: all six elements (edit until every line says PASS).
_checks = {
    "A1.1 only from sources": bool(re.search(r"only|solely|numbered sources", ANSWER_SYSTEM, re.I)),
    "A1.2 cite by index": bool(re.search(r"\[1\]|\[n\]|citation", ANSWER_SYSTEM, re.I)),
    "A1.3 never invent a number": bool(re.search(r"never cite|not (?:given|supplied)|out of range", ANSWER_SYSTEM, re.I)),
    "A1.4 exact refusal": REFUSAL in ANSWER_SYSTEM,
    "A1.5 disagreement": bool(re.search(r"disagree|conflict|contradict", ANSWER_SYSTEM, re.I)),
    "A1.6 length": bool(re.search(r"sentence|concise|brief|short", ANSWER_SYSTEM, re.I)),
}
for name, ok in _checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(_checks.values()), "TODO A1 incomplete: fix the FAIL lines above"
print("\nANSWER_SYSTEM ready (", len(ANSWER_SYSTEM), "chars )")


In [ ]:
# A2 - AFTER writing yours: diff against the reference prompt.
# Read both and note every difference in report.md (not a pass/fail check).
from aip.rag import ANSWER_SYSTEM as REF_ANSWER_SYSTEM

print("=== YOURS ===")
print(ANSWER_SYSTEM)
print("\n=== REFERENCE (aip/rag.py) ===")
print(REF_ANSWER_SYSTEM)
print("\nDiffs to note in report.md: rule order, refusal handling,")
print("partial-refusal clause, untrusted-context clause, length wording.")


## Part B - Citation enforcement (30 min)

**B1.** Number the sources so hallucinated indices are *provable* errors.

**B2.** Implement `validate_answer()` - three checks, all mechanical.

**B3.** Decide what happens when validation fails. Defend the choice. Never
return `citations_valid=False` with `refused=False`.


In [ ]:
# B1 provided: read format_context - numbering is what makes [7] with five
# sources a detectable bug instead of an invisible one.
from aip.retrieval import DenseRetriever, format_context
from labs.lab3.search import build_chunks

def build_retriever_local():
    """Your Lab 3 winner. Default: markdown-400 dense (reference config).
    Replace with YOUR winning strategy/size from reports/lab3_sweeps.json."""
    chunks = build_chunks(corpus, "markdown", 400)
    return DenseRetriever(chunks, show_progress=False)

retriever = build_retriever_local()
sample_q = questions_answerable[0]["question"]
hits = retriever.search(sample_q, k=5)
print(format_context(hits[:2]))
print("...")
print(f"n_sources for this query = {len(hits)}  -> any [n] outside 1..{len(hits)} is invalid")


In [ ]:
# YOUR TURN - B2: validate_answer. Three checks, all mechanical.
#
#   B2.1  every [n] is in 1..n_sources  (use _cited_indices or enforce_citations)
#   B2.2  not truncated (finish_reason == "length") and not empty
#   B2.3  a non-refusal answer has at least one citation
#
# Return dict with at least:
#   {"valid", "refused", "invalid_citations", "n_citations", "truncated", "reason"}
#
# A refusal needs no citation; truncation and out-of-range indices fail even
# on a refusal. valid = no problems found.

def validate_answer(text: str, n_sources: int, finish_reason: str | None = None) -> dict:
    # TODO B2.1-B2.3
    raise NotImplementedError("TODO B2: implement validate_answer (three checks)")

# --- tests: all must pass before Part B means anything ---
def _t(label, cond):
    print(("PASS " if cond else "FAIL ") + label)
    return cond

r_refuse = validate_answer(REFUSAL, 3, None)
r_ok     = validate_answer("The window is 30 days. [1]", 3, None)
r_bad    = validate_answer("The window is 30 days. [9]", 3, None)       # out of range
r_nocit  = validate_answer("The window is 30 days.", 3, None)           # no citation
r_trunc  = validate_answer("The window is 30 days. [1]", 3, "length")   # truncated
r_empty  = validate_answer("", 3, None)

oks = [
    _t("B2 refusal valid, no citation needed", r_refuse["valid"] and r_refuse["refused"]),
    _t("B2 in-range citation valid", r_ok["valid"] and r_ok["n_citations"] == 1),
    _t("B2 out-of-range citation invalid", (not r_bad["valid"]) and 9 in r_bad["invalid_citations"]),
    _t("B2 non-refusal without citation invalid", not r_nocit["valid"]),
    _t("B2 truncated invalid", (not r_trunc["valid"]) and r_trunc["truncated"]),
    _t("B2 empty invalid", not r_empty["valid"]),
]
assert all(oks), "TODO B2 incomplete: fix the FAIL tests above"
print("validate_answer: all tests passed")


In [ ]:
# YOUR TURN - B3: answer_question - retrieve -> generate -> validate -> repair.
#
#   B3.1  retrieve k, optional rerank to final_k
#   B3.2  generate via _generate(question, hits, tier=...)
#   B3.3  v = validate_answer(text, len(hits), finish)
#   B3.4  if invalid and not refused: ONE corrective retry via _generate(..., correction=...)
#         - if retry valid -> repaired=True
#         - else -> fall back to exact REFUSAL (never ship a bad citation)
#   B3.5  return Answer(...) with citations_valid = v["valid"] or v["refused"]
#
# Defend your B3.4 choice in one comment (retry / strip / refuse).
# NEVER return citations_valid=False and refused=False together.

def answer_question(question: str, retriever, *, k: int = 12,
                    final_k: int = 5, reranker=None, tier: str = "MAIN") -> Answer:
    # TODO B3.1-B3.5
    raise NotImplementedError("TODO B3: implement answer_question with a repair strategy")

# Push into the module so later cells / evaluate.py import your version:
import importlib
import labs.lab4.rag as rag_mod
rag_mod.ANSWER_SYSTEM = ANSWER_SYSTEM
rag_mod.validate_answer = validate_answer
rag_mod.answer_question = answer_question
# also patch the names evaluate.py already imported, if loaded:
import labs.lab4.evaluate as ev_mod
ev_mod.answer_question = answer_question
ev_mod.is_refusal = is_refusal
print("injected ANSWER_SYSTEM / validate_answer / answer_question into labs.lab4.*")


In [ ]:
# B3 smoke test (1-2 API calls, ~$0.001). Run once after B3 is implemented.
# Skip while developing - re-run before Part C.
try:
    a = answer_question(questions_answerable[0]["question"], retriever, k=8, final_k=5)
    print("Q:", questions_answerable[0]["id"])
    print("A:", a.text[:300])
    print("refused:", a.refused, " citations_valid:", a.citations_valid,
          " repaired:", a.repaired, " n_citations:", a.n_citations)
    assert a.citations_valid or a.refused, "B3 invariant broken: bad citation + not refused"
    print("smoke OK - invariant holds")
except NotImplementedError as exc:
    print("not ready:", exc)


## Part C - Refusal, both directions (30 min)

**C1.** Run the 5 unanswerable questions. Count refusals.

**C2. Q37 is the interesting one.** Partial answer: state what is supported,
refuse the rest. Most systems miss this on the first attempt.

**C3.** Measure the other direction - wrongful refusals on answerable questions.

**C4.** Move the dial. Stricter refusal instruction, re-run, report both
numbers. Then set it for an insurance helpdesk **with a reason about the
relative cost of the two errors** - "higher precision" is not a reason.


In [ ]:
# C1 - the 5 unanswerable questions. Costs a few cents; run once.
# Counts refusals. n=5: one case moves recall by 0.20 - report raw counts.
UNANS_IDS = {"Q36", "Q37", "Q38", "Q39", "Q40"}
unans_rows = []
try:
    for q in unanswerable:
        a = answer_question(q["question"], retriever, k=8, final_k=5)
        unans_rows.append({"id": q["id"], "refused": a.refused, "text": a.text[:120]})
        print(f"  {q['id']} refused={a.refused}  {a.text[:100]!r}")
    n_ref = sum(r["refused"] for r in unans_rows)
    print(f"\nC1: {n_ref}/{len(unans_rows)} unanswerable refused (target >= 4/5)")
    print("Report BOTH the ratio and the raw counts - n=5 is noisy.")
except NotImplementedError as exc:
    print("not ready:", exc)


In [ ]:
# C2 - Q37: partial answer. Corpus confirms a Platinum international benefit
# exists, but the addendum describing limits is missing.
# Right behaviour: answer the supported part WITH citations, refuse the rest.
q37 = next(q for q in questions_all if q["id"] == "Q37")
print("Q37:", q37["question"])
print("gold:", q37.get("gold_answer", "")[:300])
print("relevant:", q37.get("relevant_docs"))
try:
    a37 = answer_question(q37["question"], retriever, k=10, final_k=5)
    print("\nyour answer:")
    print(a37.text)
    print("\nrefused:", a37.refused, " citations_valid:", a37.citations_valid)
    partial = (not a37.refused) and ("don't have enough" in a37.text.lower()
                                     or "not enough information" in a37.text.lower())
    # partial = answered some AND declined some (heuristic; judge by eye too)
    print("\nCheck by eye: does it cite what IS supported and decline the limit?")
except NotImplementedError as exc:
    print("not ready:", exc)


In [ ]:
# C3 harness - refusal precision and recall on all 45.
# refill after C1/C4 re-runs; falls back to reports/lab4.json if present.
def refusal_rates(rows):
    """rows: list of dicts with id, refused, unanswerable."""
    una = [r for r in rows if r["unanswerable"]]
    ref = [r for r in rows if r["refused"]]
    should = sum(1 for r in una if r["refused"])
    right = sum(1 for r in ref if r["unanswerable"])
    recall = should / len(una) if una else 0.0
    precision = right / len(ref) if ref else 1.0
    return {
        "recall": recall, "precision": precision,
        "should_refused": should, "n_unanswerable": len(una),
        "n_refused": len(ref), "n_wrong_refusals": len(ref) - right,
        "raw": f"{should}/{len(una)} recall, {right}/{len(ref)} precision",
    }

# Prefer a live run if you have rows; else load the saved full eval:
lab4_path = ROOT / "reports/lab4.json"
if lab4_path.exists():
    saved = json.loads(lab4_path.read_text(encoding="utf-8"))
    rates = refusal_rates(saved)
    print("from reports/lab4.json:")
else:
    rates = None
    print("reports/lab4.json not found - run Part E1 first, or build rows from C1.")
if rates:
    for k, v in rates.items():
        print(f"  {k}: {v}")
    print("\nTargets: recall >= 4/5, precision >= 0.70. Report raw counts (n=5).")


In [ ]:
# YOUR TURN - C4: move the dial, then answer the product question.
# 1. Make the refusal instruction stricter in ANSWER_SYSTEM (or a variant).
# 2. Re-run C1/C3 (or a subset) and record both numbers.
# 3. State where you would set it for an insurance helpdesk, and WHY
#    (relative cost of false refusal vs wrong confident answer).
#
# Fill each ___ from your runs, copy into report.md C4.

C4_ANSWER = """
Strictness setting tried: base / stricter (circle one + describe the edit)
  recall   before=___ after=___   raw: ___ -> ___
  precision before=___ after=___  raw: ___ -> ___

Product recommendation for an insurance helpdesk:
  set the dial at: ___
  because (cost of error A vs error B): ...
""".strip()
print(C4_ANSWER)


## Part D - The judge (45 min)

**D1.** Two separate single-criterion rubrics - never one judge weighing
several things.

**D2. Calibration is required.** Hand-label 20 answers *before* looking at
the judge. Report κ. If κ < 0.4, fix the rubric and re-run.

**D3.** Self-preference: if judge tier == generator tier, say so and state
the direction of bias. `llm_judge` defaults to `tier="LARGE"` for a reason.


In [ ]:
# YOUR TURN - D1: two single-criterion rubrics (T3 §4.2).
# Start from aip.evals.JUDGE_RUBRIC_* if you want, then IMPROVE them.
# evaluate.py should end up with your improved versions (copy them over).
#
#   D1.1  RUBRIC_FAITHFULNESS - one score: is EVERY claim in ANSWER supported
#         by CONTEXT? (implied strength counts: limits, dates, always/never)
#   D1.2  RUBRIC_CORRECTNESS - score 0|1|2 vs REFERENCE; refusal handled:
#         REFUSE reference + refusal = 2; confident answer = 0.
#
# Each rubric: single criterion, JSON reply format, {context}/{answer} (or
# {question}/{reference}/{candidate}) placeholders.

from aip.evals import JUDGE_RUBRIC_FAITHFULNESS, JUDGE_RUBRIC_CORRECTNESS

# TODO D1.1 - improve faithfulness (start from the aip.evals template)
RUBRIC_FAITHFULNESS = JUDGE_RUBRIC_FAITHFULNESS  # replace with yours

# TODO D1.2 - improve correctness (must handle REFUSE / PARTIAL REFUSE)
RUBRIC_CORRECTNESS = JUDGE_RUBRIC_CORRECTNESS  # replace with yours

assert "{context}" in RUBRIC_FAITHFULNESS and "{answer}" in RUBRIC_FAITHFULNESS
assert "{question}" in RUBRIC_CORRECTNESS and "{reference}" in RUBRIC_CORRECTNESS
assert "{candidate}" in RUBRIC_CORRECTNESS
print("rubric placeholders OK - now edit the prose, then copy into evaluate.py")

# inject for runners below
import labs.lab4.evaluate as ev
ev.RUBRIC_FAITHFULNESS = RUBRIC_FAITHFULNESS
ev.RUBRIC_CORRECTNESS = RUBRIC_CORRECTNESS


In [ ]:
# YOUR TURN - D1 continued: the two judge functions (copy into evaluate.py).
# Parse-failure -> return None (missing data), NEVER 0.
#
#   judge_faithfulness(answer_text, context) -> 0 | 1 | None
#   judge_correctness(question, candidate, reference) -> 0 | 1 | 2 | None
#
# judge_correctness: handle refusal DETERMINISTICALLY first when the reference
# is a full REFUSE (no judge call needed). PARTIAL still goes to the judge.

from aip.evals import llm_judge

def _verdict_score(verdict: dict, lo: int, hi: int) -> int | None:
    if verdict.get("parse_error"):
        return None
    try:
        score = int(verdict.get("score"))
    except (TypeError, ValueError):
        return None
    if score < lo or score > hi:
        return None
    return score

def judge_faithfulness(answer_text: str, context: str) -> int | None:
    # TODO D1.1
    verdict = llm_judge(RUBRIC_FAITHFULNESS.format(
        context=context[:8000], answer=answer_text), tier="LARGE")
    return _verdict_score(verdict, 0, 1)

def _reference_declines(reference: str) -> str:
    head = reference.lstrip()[:40].upper()
    if head.startswith("PARTIAL REFUSE"):
        return "partial"
    if head.startswith("REFUSE"):
        return "full"
    return ""

def judge_correctness(question: str, candidate: str, reference: str) -> int | None:
    # TODO D1.2 - deterministic refusal branch first
    ref_mode = _reference_declines(reference)
    cand_refuses = is_refusal(candidate)
    if ref_mode == "full":
        return 2 if cand_refuses else 0
    if ref_mode == "" and cand_refuses:
        return 0
    verdict = llm_judge(RUBRIC_CORRECTNESS.format(
        question=question, reference=reference, candidate=candidate), tier="LARGE")
    return _verdict_score(verdict, 0, 2)

ev.judge_faithfulness = judge_faithfulness
ev.judge_correctness = judge_correctness
ev.RUBRIC_FAITHFULNESS = RUBRIC_FAITHFULNESS
ev.RUBRIC_CORRECTNESS = RUBRIC_CORRECTNESS
print("judges injected into labs.lab4.evaluate")
print("NOTE tier='LARGE' (different from generator) - self-preference (D3)")


In [ ]:
# D2 - calibration sheet. Hand-label BEFORE looking at judge scores.
# Requires reports/lab4.json (run Part E1 first if missing).
from labs.lab4.evaluate import make_calibration_sheet, report_kappa, LABEL_SHEET

lab4_path = ROOT / "reports/lab4.json"
if not lab4_path.exists():
    print("reports/lab4.json missing - run Part E1 first, then re-run this cell.")
else:
    make_calibration_sheet()
    print(f"\nOpen {LABEL_SHEET}")
    print("Fill human_faithfulness (0/1) and human_correctness (0/1/2) for all 20,")
    print("THEN re-run this cell to compute kappa.")
    try:
        report_kappa()
    except Exception as exc:
        print(f"kappa not ready yet ({type(exc).__name__}: {exc}) - label first.")


In [ ]:
# YOUR TURN - D3: self-preference note for report.md.
# Is your judge tier the same as the generator tier? What is the direction
# of the bias if so? What did you do about it?
D3_ANSWER = """
Generator tier: ___   Judge tier: ___
Same model? Y/N
Direction of bias if same: ...
What I did about it (different tier / provider / caveat in report): ...
""".strip()
print(D3_ANSWER)


## Part E - Full evaluation + decomposition (35 min)

**E1.** `--full --save reports/lab4.json` - every metric. **Lab 5 reads this file.**

**E2.** Gold-context decomposition - the highest-value 10 minutes in the lab:

```
A = correctness with gold context      <- generation ceiling
B = correctness with retrieved context <- your system
retrieval-attributable loss  = A - B
generation-attributable loss = 1 - A
```

Whichever is larger is where Lab 5 goes.

**E3.** Read 10 wrong answers. Tag each with one of the seven failure modes.
That is your Lab 5 backlog.


In [ ]:
# E1 - full evaluation. ~45 questions x (generate + 2 judges). Budget ~$1.
# Resumable: re-running continues from reports/lab4.json.
# Requires ANSWER_SYSTEM, validate_answer, answer_question, judges injected.
from labs.lab4.evaluate import run_full

try:
    # cheap preflight: one question through the pipeline
    _ = answer_question(questions_answerable[0]["question"], retriever, k=8, final_k=5)
    print("preflight OK - launching full run (this takes several minutes)")
    run_full(save="reports/lab4.json")
except NotImplementedError as exc:
    print("not ready:", exc)
except Exception as exc:
    print(f"run failed ({type(exc).__name__}: {exc}) - check Part A/B/D TODOs")


In [ ]:
# E1 report - every metric in the target table. Paste into report.md.
lab4_path = ROOT / "reports/lab4.json"
if not lab4_path.exists():
    print("reports/lab4.json not found - run E1 first.")
else:
    rows = json.loads(lab4_path.read_text(encoding="utf-8"))
    ans = [r for r in rows if not r.get("unanswerable")]
    una = [r for r in rows if r.get("unanswerable")]
    refusals = [r for r in rows if r.get("refused")]

    def mean(xs):
        xs = [x for x in xs if x is not None]
        return statistics.fmean(xs) if xs else None

    faith = mean([r.get("faithfulness") for r in rows])
    corr = mean([r.get("correctness") for r in ans])
    cit = statistics.fmean(r.get("citations_valid", False) for r in rows)
    rec = sum(1 for r in una if r.get("refused")) / len(una) if una else 0
    prec = sum(1 for r in refusals if r.get("unanswerable")) / len(refusals) if refusals else 1.0
    repair = statistics.fmean(r.get("repaired", False) for r in rows)
    n_parse = sum(1 for r in rows if r.get("faithfulness") is None)

    metrics = {
        "citation validity": (cit, 1.00),
        "faithfulness": (faith, 0.90),
        "correctness (0-2)": (corr, 1.50),
        "correctness norm": (corr / 2 if corr is not None else None, 0.75),
        "refusal recall": (rec, 0.80),
        "refusal precision": (prec, 0.70),
        "repair rate": (repair, None),
        "judge parse failures (excluded)": (float(n_parse), None),
    }
    print(f"n = {len(rows)}  ({len(ans)} answerable, {len(una)} unanswerable)")
    print(f"{'metric':<32}{'value':>10}{'target':>10}  met?")
    for name, (v, t) in metrics.items():
        if v is None:
            print(f"{name:<32}{'n/a':>10}")
            continue
        met = "" if t is None else ("yes" if v >= t else "NO")
        ts = "-" if t is None else f"{t:.2f}"
        print(f"{name:<32}{v:>10.3f}{ts:>10}  {met}")
    print("\nCopy this table into report.md E1.")
    print("If faithfulness << correctness, suspect truncated judge verdicts (parse failures).")


In [ ]:
# E2 - gold-context decomposition. Requires answer_with_gold_context (TODO E2).
from labs.lab4.evaluate import run_gold_context

# TODO E2 is in rag.py: same generator, gold docs as context, no retrieval.
# If you have not implemented it, the run will fail - implement, then re-run.
try:
    run_gold_context()
    print("\nCopy A, B, A-B, 1-A into report.md E2.")
    print("Whichever loss is larger is where Lab 5 should spend its time.")
except NotImplementedError as exc:
    print("not ready:", exc)
except Exception as exc:
    print(f"failed ({type(exc).__name__}: {exc}) - implement answer_with_gold_context (E2) first")


In [ ]:
# YOUR TURN - E3: read 10 wrong answers, tag each with a failure mode.
# Modes (T4 §5): 1 missing content, 2 chunk boundary, 3 embedding mismatch,
# 4 ranking, 5 reranker, 6 generation, 7 presentation.
# This is your Lab 5 backlog - bring it with you.
lab4_path = ROOT / "reports/lab4.json"
if not lab4_path.exists():
    print("run E1 first")
else:
    rows = json.loads(lab4_path.read_text(encoding="utf-8"))
    wrong = [r for r in rows
             if (r.get("correctness") is not None and r["correctness"] < 2)
             or not r.get("citations_valid", True)]
    print(f"{len(wrong)} wrong/uncited answers. Reading first 10 (read them!):")
    for r in wrong[:10]:
        print(f"\n{r['id']} ({r['kind']}) corr={r.get('correctness')} cit_ok={r.get('citations_valid')}")
        print("  Q:", next((q['question'] for q in questions_all if q['id']==r['id']), '?')[:100])
        print("  A:", (r.get('answer') or '')[:160])

# YOUR TURN - fill the tally, copy into report.md E3:
E3_ANSWER = """
mode 1 missing content    n = ___
mode 2 chunk boundary      n = ___
mode 3 embedding mismatch n = ___
mode 4 ranking            n = ___
mode 5 reranker           n = ___
mode 6 generation         n = ___
mode 7 presentation       n = ___
one surprising failure (id + why it surprised you): ...
""".strip()
print("\n" + E3_ANSWER)


## Submit

1. Copy your work into **`labs/lab4/rag.py`** and **`labs/lab4/evaluate.py`**:
   - `ANSWER_SYSTEM` (Part A)
   - `validate_answer` + `answer_question` (B2/B3)
   - `answer_with_gold_context` (E2)
   - both rubrics + both judges (D1)
   - `build_retriever()` = **your Lab 3 winner** (not the placeholder)
2. `reports/lab4.json` exists (E1 `--full --save`).
3. Fill `report.md` **by hand** - never regenerate it.
4. State n = 45; report **both** refusal numbers with raw counts.
5. Report κ for **both** rubrics (≥ 0.4, or say how you fixed the rubric).
6. E2 decomposition: A, B, both attributed losses.
7. E3 failure-mode tally.

Run the check cell below before you export the report.

**Also keep open:** `RUNSHEET.md` (the actions) and `README.md` (targets, rubric).


In [ ]:
# Submit check - no leftover TODOs in the graded files; lab4.json exists.
rag_src = (ROOT / "labs/lab4/rag.py").read_text(encoding="utf-8")
ev_src = (ROOT / "labs/lab4/evaluate.py").read_text(encoding="utf-8")

for name, src in [("rag.py", rag_src), ("evaluate.py", ev_src)]:
    nie = src.count("raise NotImplementedError")
    todo = len(re.findall(r"TODO [ABD]", src))
    print(f"{name}: {nie} NotImplementedError, {todo} TODO [ABD] markers")

if "markdown" in ev_src and "DenseRetriever" in ev_src:
    print("build_retriever: looks populated (verify it is YOUR lab3 winner)")
else:
    print("build_retriever: PLACEHOLDER - put your Lab 3 config here")

lab4_path = ROOT / "reports/lab4.json"
if lab4_path.exists():
    n = len(json.loads(lab4_path.read_text(encoding="utf-8")))
    print(f"reports/lab4.json: {n} rows  ({'OK' if n >= 45 else 'short - re-run E1'})")
else:
    print("reports/lab4.json MISSING - run E1")

print("\nDone when: 0 NotImplementedError in rag.py/evaluate.py, lab4.json has 45 rows,")
print("report.md filled by hand with both refusal counts + both kappas + E2 + E3.")
